Librairies

CREATION DU DATASET A PARTIR DE PAGES WIKIPEDIAS.
CREER UN FICHIER  .TXT CONTENANT DANS CHAQUE LIGNE LE TITRE DES PAGES WIKIPEDIAS A TELECHARGER et EXECUTER LA FNCTION getdumps(chemin_fichier_texte). Cela créera 2 DOSSIERS data/html et data/pages

In [ ]:
!pip install beautifulsoup4 requests

In [14]:
!pip install decorator editdistance idna networkx wikipedia-api

In [2]:
!pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 18.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.7 MB/s  0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [langchain-experimental]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [1]:
!pip install rank-bm25

In [16]:
from typing import List, Union
from random import sample
from pathlib import Path
from os import makedirs
from json import JSONEncoder, dumps
import pickle as pkl
import sys

from wikipediaapi import Wikipedia, ExtractFormat

In [17]:


def get_sections(section):
    if section.sections == []:
        return []
    else:
        return [Section(sec.title, sec.text,get_sections(sec))for sec in section.sections]



class Section:
    def __init__(self,
                 title: str,
                 text: str,
                 sections=[]):
        self.title = title
        self.text = text
        self.sections = sections

def dump_page(source: str,
              target_folder: Union[Path, str] = "pages",
              wiki_obj: Wikipedia = None,
              lang: str = 'en'):
    if not wiki_obj:
        wiki_obj = Wikipedia(lang)

    target_folder = Path(target_folder)
    if not target_folder.exists():
        makedirs(target_folder)

    wikipage = wiki_obj.page(source)
    if not wikipage.exists():
        print(f"page {source} does not exist")


    else:
        page_info = wiki_obj.info(wikipage)
        if page_info.title != wikipage.title:
            wikipage = wiki_obj.page(page_info.title)
        wiki_title = wikipage.title.replace(' ', '_')
        target_file = target_folder / (wiki_title.replace("/", "__SLASH__") + ".pkl")
        page_data = {
                    "title": wikipage.title,
                    "pageid": wikipage.pageid,
                    "fullurl": wikipage.fullurl,
                    "text":wikipage.text,
                    "summary":wikipage.summary,
                    "lastrevid":wikipage.lastrevid,
                }
        page_data['categories'] = {}
        for att,val in wikipage.categories.items():
          page_data['categories'][att] = {'lang':val.language,'id':val.pageid,'ns':val.ns}
        page_data['sections'] = []
        for sec in wikipage.sections:
          page_data['sections'].append(Section(sec.title,sec.text,get_sections(sec)))
        pkl.dump(page_data, target_file.open('wb'))


def getdumps(path_txt,language='en'):
    wiki_html = Wikipedia(user_agent='https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy',language=language, extract_format=ExtractFormat.HTML)
    wiki_page = Wikipedia(user_agent='https://foundation.wikimedia.org/wiki/Policy:Wikimedia_Foundation_User-Agent_Policy',language=language, extract_format=ExtractFormat.WIKI)
    with open(path_txt,'r',encoding='utf-8') as f:
        sources = f.readlines()
    sources = [s.strip() for s in sources]

    for s in sources:
        dump_page(s, target_folder='data/fomchtml', wiki_obj=wiki_html)
        dump_page(s, target_folder='data/fomcpages', wiki_obj=wiki_page)





In [ ]:
getdumps('data/sources.txt')

In [ ]:
import pickle as pkl
import numpy as np
import wikipediaapi
import matplotlib.pyplot as plt
import argparse

from wikipediaapi import Wikipedia
from tqdm import tqdm
from os import listdir


# We want to save the articles stats

def get_section_text(section, level=1):
    s_text = ''
    for s in section.sections:
        s_text += s.text
        s_text += '\n' + get_section_text(s, level + 1)
    return s_text


def compute_article(wiki_path, html_path):
    stats = {}
    wiki_path = wiki_path.replace(' ', '_')
    html_path = html_path.replace(' ', '_')

    with open(wiki_path, 'rb') as f:
        page = pkl.load(f)

    if html_path is not None:
        with open(html_path, 'rb') as f:
            page_html = pkl.load(f)
    try:
        stats['total_text_length'] = len(page['text'])
    except:
        print(wiki_path)
        print('TextError')
        return 'TextError'

    stats['draft_in_category'] = False
    stats['homonym_in_category'] = False

    try:
        for category in page['categories']:
            if 'Catégorie:Wikipédia:ébauche' in category:
                stats['draft_in_category'] = True
            if 'Catégorie:Homonymie' in category:
                stats['homonym_in_category'] = True
    except:
        print('CategoryError')
        return 'CategoryError'
    # Now we crawl over the text section by section
    stats['sections_length'] = []
    stats['paragraph_length_by_sections'] = []
    stats['paragraph_by_sections'] = []
    stats['summary_length'] = len(page['summary'])

    # First the summary (not included in the sections)

    try:
        summary_paragraphs = page['summary'].split('\n')
    except:
        print('SummaryError')
        return 'SummaryError'
    stats['paragraph_length_by_summary'] = [len(paragraph) for paragraph in summary_paragraphs]
    stats['paragraphs_in_summary'] = summary_paragraphs
    stats['section_titles'] = []
    # Now the real sections

    stats['is_a_year_article'] = False

    try:
        for i, section in enumerate(page['sections']):
            #print(section.title)
            if section.title in ['Voir aussi', 'Articles connexes', 'Liens externes', 'Notes et références']:
                break
            if section.title == 'Événements':
                stats['is_a_year_article'] = True

            # We check if the section contains lists

            if html_path is not None:
                if '<li>' in page_html['sections'][i].text:
                    current_section_text = ''
                else:
                    current_section_text = section.text + '\n'

                new_html_section_text = get_section_text(page_html['sections'][i])

                if '<li>' in new_html_section_text:
                    new_section_text = ''
                else:
                    new_section_text = get_section_text(section)

                section_text = current_section_text + new_section_text
            else:
                section_text = section.text + '\n' + get_section_text(section)

            stats['sections_length'].append(len(section_text))

            paragraphs = section_text.split('\n')
            paragraph_length_by_section = [len(paragraph) for paragraph in paragraphs]
            stats['paragraph_length_by_sections'].append(paragraph_length_by_section)
            stats['paragraph_by_sections'].append([paragraph for paragraph in paragraphs])
            stats['section_titles'].append(section.title)
    except:
        print('SectionError')
        return 'SectionError'
    return stats


def compute_files(wiki_path_list, folder_path, html_path):
    if html_path is not None:
        stats_dic = {wiki_path: compute_article(folder_path + '/' + wiki_path, html_path + '/' + wiki_path) for
                     wiki_path in tqdm(wiki_path_list)}
    else:
        stats_dic = {wiki_path: compute_article(folder_path + '/' + wiki_path, None) for wiki_path in
                     tqdm(wiki_path_list)}
    return stats_dic


def get_pkl_filenames_from_folder(folder_path):
    list_files = [f for f in listdir(folder_path) if '.pkl' in f]
    return list_files


def comp_stat():
    ## Required parameters

    output_dic_fn = 'data/stats_dic.pkl'
    folder_path = 'data/fomcpages'
    html_path = 'data/fomchtml'

    wiki_path_list = get_pkl_filenames_from_folder(folder_path)

    stats_dic = compute_files(wiki_path_list, folder_path, html_path)

    with open(output_dic_fn, 'wb') as f:
        pkl.dump(stats_dic, f)


In [ ]:
comp_stat()

100%|██████████| 10/10 [00:00<00:00, 1690.50it/s]


In [ ]:
import pickle as pkl
from IPython.utils import encoding
import numpy as np
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
import argparse
from typing import List
from json import JSONEncoder, dumps
from wikipediaapi import Wikipedia


class Paragraph:
    def __init__(self,
                 context: str):
        self.context = context
        self.qas = []


class Article:
    def __init__(self,
                 title: str,
                 paragraphs: List[str],
                 oldid: str):
        self.title = title
        self.paragraphs = [Paragraph(p) for p in paragraphs]
        self.oldid = oldid


class Dataset:
    class CustomEncoder(JSONEncoder):
        def default(self, o):
            return o.__dict__

    def __init__(self,
                 articles: List[Article],
                 version: str = 'frenchqa_1.0'):
        self.data = articles
        self.version = version

    def to_json(self):
        return dumps(self, indent=4, cls=self.CustomEncoder)


def check_number_paragraphs(article_stats, min_len_paragraphs=500, max_len_paragraphs=1000):
    if article_stats['total_text_length'] < min_len_paragraphs:
        return []
    # We do flatten the para by section list that is composed of a list of section, and a section being a list
    # of paragraphs. This list by section was done this way to also study context length by section
    flatten_section = [para for section in article_stats['paragraph_length_by_sections'] for para in section]
    all_paras = article_stats['paragraph_length_by_summary'] + flatten_section
    all_paras_filtered = [para for para in all_paras if para >= min_len_paragraphs and para < max_len_paragraphs]

    return all_paras_filtered


def get_number_paragraphs(stats_all_articles, min_len_paragraphs=500):
    nb_paragraphs = [len(check_number_paragraphs(article_stats, min_len_paragraphs)) for article_stats in
                     stats_all_articles.values()]
    return nb_paragraphs


def compute_min_len_paras_on_dic(article_stats, min_len_paragraphs=500, max_len_paragraphs=1000):
    article_stats['paras'] = check_number_paragraphs(article_stats, min_len_paragraphs, max_len_paragraphs)

    return article_stats


def filter_article_by_categories(article_stats, draft=False, homonym=False):
    try:
        if article_stats['homonym_in_category'] == homonym:
            if article_stats['draft_in_category'] == draft:
                return True
        return False
    except:
        return False


def filter_dic(stats, min_len_paragraphs=500, draft=False, homonym=False, max_len_paragraphs=1000):
    filtered_dic = {filename: check_number_paragraphs(stats[filename], min_len_paragraphs, max_len_paragraphs) for
                    filename in stats if filter_article_by_categories(stats[filename], draft, homonym)}
    return filtered_dic


def print_para_if_max(stats, max_para_len=8000):
    for filename in stats:
        for para in stats[filename]:
            if para > max_para_len:
                print(filename, para)


def filter_min_paras(stats_with_para_len, min_nb_paras):
    filtered_dic = {filename: stats_with_para_len[filename] for filename in stats_with_para_len if
                    len(stats_with_para_len[filename]) >= min_nb_paras}
    return filtered_dic


def get_section_text(section, level=1):
    s_text = ''
    for s in section.sections:
        s_text += s.text
        s_text += '\n' + get_section_text(s, level + 1)
    return s_text


def filter_years_articles(page_pkl_fn):
    # If 'Evenements' is in sections title, then it means it is a year article.
    with open(page_pkl_fn, 'rb') as f:
        page = pkl.load(f)

    for section in page['sections']:
        if section.title in ['Événements']:
            return False
        return True


def get_section_paragraphs_text(page_pkl_fn, min_len_para=500, max_len_para=1000, wiki_path=None, html_path=None):
    if wiki_path is None:
        wiki_path = ''
    with open(wiki_path + '/' + page_pkl_fn, 'rb') as f:
        page = pkl.load(f)

    if html_path is not None:
        with open(html_path + '/' + page_pkl_fn, 'rb') as f:
            page_html = pkl.load(f)

    paragraphs = [paragraph for paragraph in page['summary'].split('\n') if
                  len(paragraph) >= min_len_para and len(paragraph) < max_len_para]
    for i, section in enumerate(page['sections']):
        if section.title in ['Voir aussi', 'Articles connexes', 'Liens externes', 'Notes et références']:
            break

        # We check if the section contains lists
        if html_path is not None and '<li>' in page_html['sections'][i].text:
            current_section_text = ''
            new_html_section_text = get_section_text(page_html['sections'][i])
        else:
            current_section_text = section.text + '\n'
            new_html_section_text = current_section_text

        if html_path is not None and '<li>' in new_html_section_text:
            new_section_text = ''
        else:
            new_section_text = get_section_text(section)

        section_text = current_section_text + new_section_text

        for paragraph in section_text.split('\n'):
            if len(paragraph) >= min_len_para and len(paragraph) < max_len_para:
                paragraphs.append(paragraph)
    return paragraphs


def get_filtered_complete_dic(pkl_with_stats_fn, min_paragraphs=5, min_len_paragraphs=500, max_len_paragraphs=1000,
                              draft=False, homonym=False, years=False, wiki_path=None, clean_duplicates=False):
    with open(pkl_with_stats_fn, 'rb') as f:
        stats_uncleaned = pkl.load(f)

    # We filter out the sections errors
    stats = {key: stats_uncleaned[key] for key in stats_uncleaned if stats_uncleaned[key] != 'SectionError'}

    filtered_stats = filter_dic(stats, min_len_paragraphs=min_len_paragraphs, draft=draft, homonym=homonym,
                                max_len_paragraphs=max_len_paragraphs)
    filtered_stats = filter_min_paras(filtered_stats, min_paragraphs)

    # We filter the years

    if clean_duplicates:
        if wiki_path is None:
            print("Error : give a wikipath for duplicates cleaning")
            return
        new_ft_stats = {}
        wiki_obj = Wikipedia('fr')
        for filename, stats in filtered_stats.items():
            try:
                with open(wiki_path + '/' + filename, 'rb') as f:
                    page = pkl.load(f)
            except FileNotFoundError:
                print("Not found :" + filename)
                continue
            page_info = wiki_obj.info(page)
            new_title = title = page_info.title
            new_title = new_title.replace(' ', '_')
            new_title += '.pkl'
            new_ft_stats[new_title] = stats
        filtered_stats = new_ft_stats

    if not years:
        print("Length before year fitering :", len(filtered_stats))
        if wiki_path is None:
            filtered_stats = {filename: filtered_stats[filename] for filename in filtered_stats if
                              filter_years_articles(filename)}
        else:
            filtered_stats = {filename: filtered_stats[filename] for filename in filtered_stats if
                              filter_years_articles(wiki_path + filename)}
    print("Final length : ", len(filtered_stats))
    return filtered_stats


def getdatajson():
    ## Required parameters
    pkl_stats_dic_fn = 'data/stats_dic.pkl'

    output_json_article_fn = 'data/articles.json'
    nb_articles_to_print = None

    min_paragraphs = 5
    min_len_paragraphs = 500
    max_len_paragraphs = 1000


    wiki_path = 'data/fomcpages'
    html_path = 'data/fomchtml'

    stats = get_filtered_complete_dic(pkl_stats_dic_fn, min_paragraphs=min_paragraphs,
                                      min_len_paragraphs=min_len_paragraphs,
                                      max_len_paragraphs=max_len_paragraphs, draft=False, homonym=False,
                                      years=True, wiki_path=wiki_path, clean_duplicates=False)

    if output_json_article_fn is not None:

        articles_filename = list(stats.keys())
        random.shuffle(articles_filename)

        if nb_articles_to_print is not None:
            articles_filename = articles_filename[:nb_articles_to_print]

        articles_list = []
        for article_fn in tqdm(articles_filename):
            try:
                paragraphs = get_section_paragraphs_text(article_fn, min_len_para=min_len_paragraphs,
                                                         max_len_para=max_len_paragraphs, wiki_path=wiki_path,
                                                         html_path=html_path)
                filename = article_fn.split('/')[-1]
            except FileNotFoundError:
                continue
                # File may have been deleted already because it was a duplicate

            with open(wiki_path + '/' + filename, 'rb') as f:
                page = pkl.load(f)

            filename = filename.replace('_', ' ')
            filename = filename.replace('.pkl', '')

            articles_list.append(Article(filename, paragraphs, oldid=str(page['lastrevid'])))

        dataset = Dataset(articles_list)

        with open(output_json_article_fn, 'w', encoding='utf-8') as f:
            f.write(dataset.to_json())


In [ ]:
getdatajson()

Final length :  9


100%|██████████| 9/9 [00:00<00:00, 943.18it/s]


Une fois le fichier articles.json obtenu au format SQuad. On le passe a un LLM pour générer des questions

Storage Dataset


In [3]:
!pip install langchain-huggingface

In [7]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
encode_kwargs = {'normalize_embeddings': True}
embed_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    encode_kwargs=encode_kwargs
)

/tmp/ipykernel_2749/1423924778.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
semantic_chunker = SemanticChunker(embed_model, breakpoint_threshold_type="percentile")

In [ ]:
import psycopg2
import json
import os
import numpy as np
import sys
import argparse
from pathlib import Path
from sentence_transformers import SentenceTransformer
import spacy
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
load_dotenv()


CHUNK_SIZE = 300     # caractères
CHUNK_MIN = 256
CHUNK_MAX = 512
CHUNK_OVERLAP = 50
EMBED_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"

def chunk_text(text: str) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + CHUNK_SIZE].strip())
        start += CHUNK_SIZE - CHUNK_OVERLAP
    return [c for c in chunks if c]


def store_vector(Chunks, embeddings, tabname,language='english',supabase=False):
    if not supabase:
        conn = psycopg2.connect(
            dbname=os.getenv('DB_NAME'),
            user=os.getenv('DB_USER'),
            host=os.getenv('DB_HOST'),
            password=os.getenv('DB_PASSWORD'),
            sslmode='require',
            port=os.getenv('DB_PORT')
        )
    else:
        conn = psycopg2.connect(os.getenv('SUPABASE_URL'))
    cur = conn.cursor()
    cur.execute(f"""
    CREATE TABLE IF NOT EXISTS {tabname}(
        id SERIAL PRIMARY KEY,
        paragraph_id INT,
        title TEXT,
        content TEXT,
        chunk_start INT,
        chunk_end INT,
        embedding vector(768)
    );
    """)

    conn.commit()
    data = [(c['paragraph_id'],c['source'], c['text'],c['chunk_start'], c['chunk_end'], e) for c,e in zip(Chunks, embeddings)]

    cur.executemany(
        f"""
        INSERT INTO {tabname} (paragraph_id,title,content,chunk_start,chunk_end, embedding)
        VALUES (%s, %s, %s, %s, %s, %s)
        """,
        data
    )

    conn.commit()

    cur.execute(f"""
    CREATE INDEX ON {tabname}
    USING ivfflat (embedding vector_cosine_ops);
    """)

    conn.commit()

    cur.execute(f"""
        CREATE INDEX ON {tabname}
        USING GIN (to_tsvector('{language}', content));
        """)

    cur.execute(f"ANALYZE {tabname};")
    conn.commit()
    conn.close()

    print("Data stored successfully in the database.")

if __name__ == "__main__":
    """
    path_json_dataset est le chemin du dataset avec les questions reponses generees par un LLM (Claude sonet moyen)
    """
    path_json_dataset = "data/articles_with_qas.json"
    token_type = None
    tabname = 'WIKIDOCS'
    use_spacy = False
    ch_id = 0
    if use_spacy:
        nlp = spacy.load('en_core_web_sm') #'fr_core_news_sm'
    else:
        #splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE,chunk_overlap=CHUNK_OVERLAP)
        semantic_chunker = SemanticChunker(embed_model, breakpoint_threshold_type="percentile",
                                           add_start_index=True)
    #model = SentenceTransformer(EMBED_MODEL)
    with open(path_json_dataset, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    N_articles = len(dataset['data'])
    for plll in range(N_articles):
        article = dataset['data'][plll]
        print(f"Processing article: {article['title']}")
        title = article['title']
        metadatas = [{"paragraph_id":i} for i in range(len(article['paragraphs']))]
        semantic_chunks = semantic_chunker.create_documents([p['context'] for p in article['paragraphs']],metadatas=metadatas)
        texts = [{'source':title, 'text': chk.page_content, 'chunk_size':len(chk.page_content),
                    'chunk_start':chk.metadata['start_index'],'paragraph_id':chk.metadata['paragraph_id'],
                    'chunk_end':chk.metadata['start_index']+len(chk.page_content),
                    'ch_id':ichk+1+ch_id} for ichk,chk in enumerate(semantic_chunks)]
        ch_id = texts[-1]['ch_id']
        chunks = [c.page_content for c in semantic_chunks]
        embeddings = embed_model.embed_documents(chunks)
        store_vector(texts, embeddings, tabname)
        print(f"Article '{title}' processed and stored in the database.")
        #print(f"Article '{title}' processed and stored in the database.")
        nparags = len(article['paragraphs'])
        for npa in range(nparags):
            ch_sizes = [(i,c['chunk_start'],c['chunk_end']) for i,c in enumerate(texts) if c['paragraph_id']==npa]
            paragraph = article['paragraphs'][npa]
            nqas = len(paragraph['qas'])
            for pl in range(nqas):
                qas = paragraph['qas'][pl]
                start = qas['answers'][0]['answer_start']
                size = len(qas['answers'][0]['text'])
                min_c = [c for c in ch_sizes if c[1]<=start][-1][0]
                max_c = [c for c in ch_sizes if c[2]>=start+size-1]
                if max_c:
                    max_c = max_c[0][0]
                else:
                    max_c = len(chunks)-1
                sols = [c['ch_id'] for c in texts[min_c:max_c+1]]
                qas['answers'][0]['ch_ids'] = sols
                paragraph['qas'][pl] = qas
            article['paragraphs'][npa] = paragraph
        dataset['data'][plll] = article



Processing article: Taylor rule
Data stored successfully in the database.
Article 'Taylor rule' processed and stored in the database.
Processing article: Federal Reserve Act
Data stored successfully in the database.
Article 'Federal Reserve Act' processed and stored in the database.
Processing article: Macroeconomics
Data stored successfully in the database.
Article 'Macroeconomics' processed and stored in the database.
Processing article: Central bank
Data stored successfully in the database.
Article 'Central bank' processed and stored in the database.
Processing article: Monetary policy of the United States
Data stored successfully in the database.
Article 'Monetary policy of the United States' processed and stored in the database.
Processing article: Money creation
Data stored successfully in the database.
Article 'Money creation' processed and stored in the database.
Processing article: Federal Reserve
Data stored successfully in the database.
Article 'Federal Reserve' processed an

In [ ]:
with open(path_json_dataset.split('.')[0]+'v2.json', 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)


In [ ]:
for plll in range(1,N_articles):
  article = dataset['data'][plll]
  print(f"Processing article: {article['title']}")
  nparags = len(article['paragraphs'])
  for npa in range(nparags):
      #ch_sizes = [(i,c['chunk_start'],c['chunk_end']) for i,c in enumerate(texts) if c['paragraph_id']==npa]
      paragraph = article['paragraphs'][npa]
      nqas = len(paragraph['qas'])
      for pl in range(nqas):
        qas = paragraph['qas'][pl]
        qas['answers'][0]['ch_ids'] = [ccc_id-1 for ccc_id in qas['answers'][0]['ch_ids']]

In [ ]:
len(semantic_chunks)

53

In [11]:
import psycopg2
import re
import numpy as np
from rank_bm25 import BM25Okapi
#from langchain_cohere import CohereRerank
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModel
import requests
import os
import json
from dotenv import load_dotenv
load_dotenv()

def tokenize(text):
    return re.findall(r'\w+', text.lower())

def rank_cohere(query,texts, top_k):
  response = requests.post(
    url="https://openrouter.ai/api/v1/rerank",
    headers={
      "Authorization": f"Bearer {os.getenv('COHERE_KEY')}",
      "Content-Type": "application/json"
    },
    data=json.dumps({
      "model": "cohere/rerank-4-fast",
      "query": query,
      "documents": texts,
      "top_n": top_k
    })
  )
  results = response.json()
  indexs = [result['index'] for result in results["results"]]
  scores = [result['relevance_score'] for result in results["results"]]
  return indexs, scores
model_lists_1 = ["BAAI/bge-reranker-v2-m3",
                 "Alibaba-NLP/gte-reranker-modernbert-base",
                 "mixedbread-ai/mxbai-rerank-xsmall-v1",
                 "jinaai/jina-reranker-v2-base-multilingual",
                 "cross-encoder/ms-marco-MiniLM-L-6-v2"]
model_lists_2 = []
def rrf(rankings: list[list[int]], k: int = 60, weights=None, top_n: int = 10):
    """rankings : une liste d'identifiants triés par pertinence décroissante, par méthode."""
    scores: dict[int, float] = {}
    for i, ranking in enumerate(rankings):
        w = weights[i] if weights else 1.0
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + w / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])[:top_n]   # [(doc_id, score), ...]

def rank(query,texts,top_k,model,model_name):
    if model_name=='cohere':
      return rank_cohere(query,texts,top_k)
    elif model_name in model_lists_1:
      tokenized_query = tokenize(query)
      pairs = [[query, text] for text in texts]
      scores = model.predict(pairs)
      top_indices = np.argsort(scores)[::-1][:top_k]
      return top_indices, [scores[i] for i in top_indices]
    elif model_name in model_lists_2:
      results = model.rerank(query, texts)
      indexs = [result['index'] for result in results["results"]]
      scores = [result['relevance_score'] for result in results["results"]]
      return indexs, scores
    else:
      raise ValueError(f"Model {model_name} not supported")
    # print(f"Index: {result['index']}, Score: {result['relevance_score']}")
    # print(f"  Document: {result['document']['text']}")





In [5]:
import psycopg2
import json
import os
import numpy as np
from dotenv import load_dotenv
import pickle
import sys
#from rerank import tokenize
from rank_bm25 import BM25Okapi
load_dotenv()

def savebm25(tabname,model_folder_path):
    os.makedirs(model_folder_path,exist_ok=True)
    conn = psycopg2.connect(
            dbname=os.getenv('DB_NAME'),
            user=os.getenv('DB_USER'),
            host=os.getenv('DB_HOST'),
            password=os.getenv('DB_PASSWORD'),
            sslmode='require',
            port=os.getenv('DB_PORT')
        )

    cur = conn.cursor()
    cur.execute(f"""
        SELECT id,title,content from {tabname}
    """, )
    rows = cur.fetchall()
    documents = [r[2] for r in rows]
    tokenized_corpus = [tokenize(text) for text in documents]
    bm25 = BM25Okapi(tokenized_corpus)
    with open(os.path.join(model_folder_path,f"{tabname}_bm25.pkl"), "wb") as f:
        pickle.dump(bm25, f)
if __name__ == "__main__":
    tabname = "WIKIDOCS"
    model_folder_path = "models"
    # if len(sys.argv)>1:
    #     tabname = sys.argv[1]
    # if len(sys.argv)>2:
    #     model_folder_path=sys.argv[2]
    savebm25(tabname,model_folder_path)

In [12]:
import json
import numpy as np
import os
import pickle
import sys
import psycopg2
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModel
from rank_bm25 import BM25Okapi
from dotenv import load_dotenv
#from rerank import rank, rrf, tokenize, model_lists_1, model_lists_2
load_dotenv()

TEST_SIZE = 0.8
TOP_K = 15
CHUNK_SIZE = 300     # caractères
CHUNK_MIN = 256
CHUNK_MAX = 512
CHUNK_OVERLAP = 50
def mparam(e,q,i):
     if i==0:
        return (e,e,)



def testpipeline(questions,answers,embeddings,
        rrk_model,name,tabname,
        method,sqlq1,sqlq2,model,TOP_K):
    os.makedirs(f'data/test/{tabname}',exist_ok=True)
    queries = [q['question'] for q in questions]
    qids = [q['qid'] for q in questions]
    #extractions = {tabname: {RETRIEVAL:{qid:[] for qid in qids[:N]}  for RETRIEVAL in RETRIEVALS[:N]} for tabname in tabs}
    if method=="bm25" or method=='hybride':
        with open(f"models/{tabname}_bm25.pkl", "rb") as f:
            bm25 = pickle.load(f)
    results = []
    print(f"method {method}")
    conn = psycopg2.connect(
            dbname=os.getenv('DB_NAME'),
            user=os.getenv('DB_USER'),
            host=os.getenv('DB_HOST'),
            password=os.getenv('DB_PASSWORD'),
            sslmode='require',
            port=os.getenv('DB_PORT')
        )

    cur = conn.cursor()
    N = len(queries)
    for i,query,qid in zip(list(range(N)),queries,qids):
        print(f"query {i+1}/{N}")
        embed = embeddings[i]
        if method == 'vectorielle' or method == 'hybride':
            cur.execute(sqlq1, (embed,embed,))

            rows = cur.fetchall()
            rows = {r[0]:(r[1],r[2],r[4]) for r in rows}
            if method == 'hybride':
                vectorielle = rows
        if method == 'bm25' or method == 'hybride':
            cur.execute(sqlq2)
            rows = cur.fetchall()
            tokenized_corpus = [tokenize(r[2]) for r in rows]
            tokenized_query = tokenize(query)
            bmscores = bm25.get_scores(tokenized_query)
            keeps = np.argsort(bmscores)[::-1][:TOP_K]
            rows = {rows[i][0]:(rows[i][1],rows[i][2],bmscores[rows[i][0]-1]) for i in keeps}
            if method == 'hybride':
                bmlist = rows
        if method == 'hybride':
            vect_id = [r for r in vectorielle.keys()]
            bm_id = [r for r in bmlist.keys()]
            rankings = [vect_id,bm_id]
            rankings = rrf(rankings,top_n=TOP_K)
            rows = {}
            for doc,score in rankings:
                if doc in vect_id:
                    rows[doc] = (vectorielle[doc][0],vectorielle[doc][1],score)
                else:
                    rows[doc] = (bmlist[doc][0],bmlist[doc][1],score)
        if rrk_model == 'norerank':
            indices = list(range(TOP_K))
            scores = [1/(60+sco+1) for sco in range(len(indices))]
        else:
            indices, scores= rank(query,[r[1] for rid,r in rows.items()],TOP_K,model,rrk_model)
        solutions = []
        row_idxs = list(rows.keys())
        for idx, score in zip(indices,scores):
            row_id = row_idxs[idx]
            row = rows[row_id]
            sol = {"title":row[0],
                "id":row_id,
                "content":row[1],
                "score_retrieve":float(row[2]),
                "score_reranking":float(score)}
            solutions.append(sol)
        results.append({
            'qid':qid,
            'question':query,
            'extractions':solutions,
            'solution':answers[i]})

    with open(f'data/test/{tabname}/{name}-{method}.json','w',encoding='utf-8') as jsol:
        jsol.write(json.dumps(results, indent=4, ensure_ascii=False))

if __name__ == "__main__":
    MODEL_NAMES = {"cross-encoder/ms-marco-MiniLM-L-6-v2":"marco","norerank":"norerank","Alibaba-NLP/gte-reranker-modernbert-base":"gte",
                 "mixedbread-ai/mxbai-rerank-xsmall-v1":"mxbai"}
                 #"jinaai/jina-reranker-v2-base-multilingual":"jina"}#"cohere":"cohere","BAAI/bge-reranker-v2-m3":"bge"
                    # }#"jinaai/jina-reranker-v3":"jina"}
    dataset_json_path = 'data/articles_with_qasv2.json'
    methods = ['hybride','bm25','vectorielle']
    #rrk_model = 'norerank'
    tabname = 'WIKIDOCS'
    # if len(sys.argv)>1:
    #     dataset_json_path = sys.argv[1]
    # if len(sys.argv)>2:
    #     method = sys.argv[2]
    # if len(sys.argv)>3:
    #     rrk_model = sys.argv[3]
    # try:
    #     name = MODEL_NAMES[rrk_model]
    # except:
    #     raise ValueError("Modèle de Reranking doit être dans MODEL_NAMES")
    with open(dataset_json_path, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    questions = []
    answers = []
    for article in dataset['data']:
            print(f"Processing article: {article['title']}")
            title = article['title']
            chunks = []
            for paragraph in article['paragraphs']:
                for qas in paragraph['qas']:
                    questions.append({'question':qas['question'],
                                    'qid':qas['id']})
                    answers.append(qas['answers'][0])
    rng = np.random.default_rng(seed=42)
    idxs = list(range(len(questions)))
    rng.shuffle(idxs)
    i = 20
    print(questions[i])
    query = questions[i]
    questions = [questions[i] for i in idxs]
    answers = [answers[i] for i in idxs]

    N = int(len(questions) * TEST_SIZE)
    N = 120

    #embeddings = encoder.encode([q['question'] for q in questions[:N]], normalize_embeddings=True).tolist()
    embeddings = embed_model.embed_documents([q['question'] for q in questions[:N]])
    for rrk_model,name in MODEL_NAMES.items():
      if rrk_model in model_lists_1:
          model = CrossEncoder(rrk_model)
      elif rrk_model in model_lists_2:
          model = AutoModel.from_pretrained(rrk_model)
      else:
          model = None
      sql1 = f"""
          SELECT
              id,
              title,
              content,
              ROW_NUMBER() OVER (
                          ORDER BY embedding <=> %s::vector
                      ) AS rank,
              embedding <=> %s::vector AS distance
          FROM {tabname}
          ORDER BY rank ASC LIMIT {TOP_K};
          """
      sql2 = f"""
          SELECT
              id,
              title,
              content
          FROM {tabname};
          """
      for method in methods:
        testpipeline(questions[:N],answers[:N],
                  embeddings,rrk_model,name,
                  tabname,method,sql1,sql2,model,
                  TOP_K)

Processing article: Taylor rule
Processing article: Federal Reserve Act
Processing article: Macroeconomics
Processing article: Central bank
Processing article: Monetary policy of the United States
Processing article: Money creation
Processing article: Federal Reserve
Processing article: Federal Open Market Committee
Processing article: Open market operation
{'question': 'What constant restriction does the standard Taylor rule impose on response coefficients?', 'qid': '1374722503-10-0'}


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

method hybride
query 1/120
query 2/120
query 3/120
query 4/120
query 5/120
query 6/120
query 7/120
query 8/120
query 9/120
query 10/120
query 11/120
query 12/120
query 13/120
query 14/120
query 15/120
query 16/120
query 17/120
query 18/120
query 19/120
query 20/120
query 21/120
query 22/120
query 23/120
query 24/120
query 25/120
query 26/120
query 27/120
query 28/120
query 29/120
query 30/120
query 31/120
query 32/120
query 33/120
query 34/120
query 35/120
query 36/120
query 37/120
query 38/120
query 39/120
query 40/120
query 41/120
query 42/120
query 43/120
query 44/120
query 45/120
query 46/120
query 47/120
query 48/120
query 49/120
query 50/120
query 51/120
query 52/120
query 53/120
query 54/120
query 55/120
query 56/120
query 57/120
query 58/120
query 59/120
query 60/120
query 61/120
query 62/120
query 63/120
query 64/120
query 65/120
query 66/120
query 67/120
query 68/120
query 69/120
query 70/120
query 71/120
query 72/120
query 73/120
query 74/120
query 75/120
query 76/120
query 

In [18]:
titles = [d['title'] for d in dataset['data']]
titles

['Taylor rule',
 'Federal Reserve Act',
 'Macroeconomics',
 'Central bank',
 'Monetary policy of the United States',
 'Money creation',
 'Federal Reserve',
 'Federal Open Market Committee',
 'Open market operation']

In [ ]:
emb = embed_model.embed_documents(chunks)

In [ ]:
smc  = semantic_chunks[0]

In [ ]:
smc.

'A central bank, reserve bank, national bank, state bank, or monetary authority is a financial institution that manages the monetary policy of a country or monetary union. In contrast to a commercial bank, a central bank possesses a monopoly on increasing the monetary base.'

In [ ]:
len(emb[0])

768

In [ ]:
chunks

['A central bank, reserve bank, national bank, state bank, or monetary authority is a financial institution that manages the monetary policy of a country or monetary union. In contrast to a commercial bank, a central bank possesses a monopoly on increasing the monetary base.',
 'Many central banks also have supervisory or regulatory powers to ensure the stability of commercial banks in their jurisdiction, to prevent bank runs, and, in some cases, to enforce policies on financial consumer protection, and against bank fraud, money laundering, or terrorism financing. Central banks play a crucial role in macroeconomic forecasting, which is essential for guiding monetary policy decisions, especially during times of economic turbulence.',
 "The notion of central banks as a separate category from other banks has emerged gradually, and only fully coalesced in the 20th century. In the aftermath of World War I, leading central bankers of the United Kingdom and the United States respectively, Mon

In [ ]:
semantic_chunks = semantic_chunker.create_documents([d.page_content for d in documents])